In [ ]:
import numpy as np
from IPython.display import Audio
from scipy.signal import sawtooth, square, butter, filtfilt

In [176]:
#Converts simbolic note value to frequency

def conversion(note):

    notes = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
    
    if note == 'rest':
        freq = 0
        
    else:
        note_letter = note[:-1] #Extract Note 
        
        octave = int(note[-1]) #Extract Octave
    
        degree = notes.index(note_letter) #Obtain # of semitones away from C
    
        midi = degree + (octave + 1) * 12 #Calculate the midi value
    
        freq = 440 * 2 ** ((midi - 69) / 12) #Calculate the frequency
    
    return freq

#Converts simbolic duration to miliseconds

def dur_to_ms(durations, bpm, beat_val = 4):
    beat_ms = 60000 / bpm
    notes_ms = beat_ms / durations * beat_val

    return notes_ms

In [177]:
#ADSR Envelope

def envelope(x, a = 2000, d = 1000, s = 1, r = 20, a_max = 1, sr = 44100):
  
    # make sure release has minnimum of 20ms.
    if r < 20:
        r = 20
    
    # transfer ms to number of samples.
    a_samples = int(a * sr / 1000) 
    d_samples = int(d * sr / 1000)
    r_samples = int(r * sr / 1000)
    
    # calculate nummber of sustain samples by subtracting numbers of samples of a, d, r from length of x (minimum value of 0).
    s_samples = max(len(x) - a_samples - d_samples - r_samples, 0)
    
    # create the distination array that has same length as x.
    envelope = np.zeros(len(x))
    
    # create a ramp from 0 to the max amplitude and set a_samples of distination array to the ramp.
    if a_samples > 0:
        envelope[:a_samples] = np.linspace(0, a_max, a_samples)
    
    # when sustain is not False.
    if s != False:
    
        # create a ramp from max amplitude to sustain level and set d_samples of distination array to the ramp.
        decay_end = a_samples + d_samples
        if d_samples > 0:
            envelope[a_samples:decay_end] = np.linspace(a_max, s, d_samples)
    
        # set the s_samples of distination array to the ramp.
        if s_samples > 0:
            envelope[decay_end:decay_end + s_samples] = s
    
        # create a ramp from sustain level to 0 and set r_samples of distination array to the ramp.
        release_start = decay_end + s_samples
        release_end = release_start + r_samples
        if r_samples > 0 and release_start < len(x):
            envelope[release_start:release_end] = np.linspace(s, 0, r_samples)
     
    # when sustain is False
    else:
        # create a ramp from max amplitude to 0 and set r_samples of distination array to the ramp.
        release_end = a_samples + r_samples
        if r_samples > 0 and a_samples < len(x):
            envelope[a_samples:release_end] = np.linspace(a_max, 0, r_samples)
    
    
    return x * envelope

In [178]:
#Generate simple waveforms

def genWave(wavetype, freq, t, sr = 44100):
    
    time_array = np.linspace(0, t / 1000, int(sr * (t / 1000)))
    
    if (wavetype == 'sinusoid'):

        out = np.sin(2 * np.pi * freq * time_array) #Generate Sinusoid
    
    elif (wavetype == 'sawtooth'):
        
        out = sawtooth(2 * np.pi * freq * time_array) #Generate Sawtooth
        
    
    elif (wavetype == 'square'):

        out = square(2 * np.pi * freq * time_array) #Generate Square
        
    
    elif (wavetype == 'triangle'):

        out = sawtooth(2 * np.pi * freq * time_array, width = 0.5) #Generate Triangle 
        
    else:
        print("Unsupported Waveform")
        return None
        
    return out


#Generate waves for chords

def genChord(notes, t, waveshape, sr = 44100):
    
    chord_wave = np.zeros(int(sr * (t / 1000)))
    
    for note in notes: #Use additive synth to stack up each notes
        
        freq = conversion(note) #Convert the symbolic pitch to frequency
        
        wave = genWave(waveshape, freq, t) #Create the wave
                
        chord_wave += wave #Add the waves together
        
    chord_wave /= len(notes) #Normalize 
    
    return chord_wave

In [179]:
#FM Function

def FM(I, carrier_frequency, modulator_frequency, t, car_shape = 'sinusoid', sr = 44100):
    
    time_array = np.linspace(0, t / 1000, int(sr * (t / 1000))) #Time array
    
    modulator = I * np.sin(2 * np.pi * modulator_frequency * time_array) #Modulator Signal
    
    if car_shape == 'sinusoid':
        product = np.sin(2 * np.pi * carrier_frequency * time_array + modulator) #Sinusoid Frequency Modulated
    
    elif car_shape == 'square':
        product = square(2 * np.pi * carrier_frequency * time_array + modulator) #Square Frequency Modulated
    
    elif car_shape == 'sawtooth':
        product = sawtooth(2 * np.pi * carrier_frequency * time_array + modulator) #Sawtooth Frequency Modulated
    
    elif car_shape == 'triangle':
        product = sawtooth(2 * np.pi * carrier_frequency * time_array + modulator, width = 0.5) #Sawtooth Frequency Modulated
    
    return product

In [180]:
#Delay Function

def Delay(x, offset, repeats, A, fs = 44100):
    
    copy = x.copy() # Copy the input data
    iters = [] #Create the output array
    
    for i in range(repeats + 1):
        
        pad_beg = np.zeros(int(fs / 1000 * offset * i)) # Create the beggining pad
        pad_end = np.zeros(int(fs / 1000 * offset * (repeats - i))) # Create the ending pad
        delay = A * np.concatenate([pad_beg, copy, pad_end]) #Concatenate the pads and the copy of the input data
        
        iters.append(delay) #Add it to the output array 
    
    out = np.sum(np.array(iters),axis = 0) 
    len_pad = len(out) - len(x)
    pad = np.zeros(len_pad)
    x = np.append(x, pad)

    return x + out

In [181]:
#Iterates through the melody array

def song_iterater(song_array, bpm, waveshape, fm, I, Mod, a, d, s, r):
    
    song = []
    
    for dur, note in song_array: 
        
        freq = conversion(note) #Convert the symbolic pitch to frequency
        t = dur_to_ms(dur, bpm) #Convert the symbolic duration to milliseconds
        
        if fm == False:
            wave = genWave(waveshape, freq, t) #Create the wave

        else: #Call Frequency Modulation Function if chosen
            wave = FM(1, freq, Mod, t, car_shape = waveshape)
        
        wave = envelope(wave, a, d, s, r) #Call Envelope Function to eliminate clips
        
        song = np.concatenate((song, wave)) #Add the wave to the output array for the melody

    return song

In [182]:
#Iterates through the chords array

def chord_iterater(chord_array, bpm, waveshape, a, d, s, r):
    
    chords = []
    
    for dur, chord in chord_array: 
        
        t = dur_to_ms(dur, bpm) #Convert the symbolic duration to milliseconds
        wave = genChord(chord, t, waveshape) #Create the waves for the chord
        wave = envelope(wave, a, d, s, r) #Call Envelope Function to eliminate clips
        chords = np.concatenate((chords, wave)) #Add the wave to the output array for the chords
    
    return chords

In [184]:
#Main Synthesizer 

def player(melody, melody_2, chord_data, bpm = 120, #Data and Tempo
           melody_waveshape = 'sinusoid', melody2_waveshape = 'sinusoid',chord_waveshape = 'sinusoid',  #Waveshapes of data
           a = 20, d = 20, s = 1, r = 30,                                                              #Envelope Values
           fm = False, I = 1, Mod = 0,                                                                 #FM Values
           delay = False, offset_ms = 300, repeat = 1, delay_amplitude = 1,                            #Delay Values 
           midi = False):                          
    
    if midi == False:
        song_1 = song_iterater(melody, bpm, melody_waveshape, fm, I, Mod, a, d, s, r) #Go through the melody array
        song_2 = song_iterater(melody_2, bpm, melody2_waveshape, fm, I, Mod, a, d, s, r) #Go through the melody2 array
    
    max_length = max(len(song_1), len(song_2)) # Pad the smaller array to match the length (melody1 and melody2)  
    if len(song_1) < max_length:
        song_1 = np.pad(song_1, (0, max_length - len(song_1)))
    if len(song_2) < max_length:
        song_2 = np.pad(song_2, (0, max_length - len(song_2)))
    
    song = song_1 + song_2 #Add 2 melodies 
    
    chords = chord_iterater(chord_data, bpm, chord_waveshape, a, d, s, r) #Go through the chord array
    
    if delay == True: #Call Delay Function if chosen
        song = Delay(song, offset_ms, repeat, delay_amplitude)
    
    max_length = max(len(song), len(chords)) # Pad the smaller array to match the length (song and chords)  
    if len(song) < max_length:
        song = np.pad(song, (0, max_length - len(song)))
    if len(chords) < max_length:
        chords = np.pad(chords, (0, max_length - len(chords)))
    
    out = song + chords #Add melodies and chords together
    
    return Audio(out, rate = 44100)

In [185]:
#Demo 1

melody = [(8, 'G3'), (8, 'B3'), (2, 'E4'), (8, 'G3'), (8, 'B3'), (2, 'F#4'), (8, 'G3'), (8, 'B3'), (2, 'G4'),
         (8, 'G3'), (8, 'B3'), (2, 'F#4'), (8, 'G3'), (8, 'B3'), (4, 'E4'), (4, 'rest'), (8, 'G3'), (8, 'B3'), 
         (4, 'F#4'), (4, 'rest'), (8, 'G3'), (8, 'B3'), (4, 'G4'), (4, 'rest'), (8, 'G3'), (8, 'B3'), (4, 'F#4'),
         (4, 'rest'), (8, 'B3'), (8, 'D4'), (4, 'A4'), (4, 'rest'), (8, 'A#3'), (8, 'D4'), (4, 'G4'), (4, 'rest'),
         (8, 'A3'), (8, 'C4'), (4, 'G4'), (4, 'rest'), (8, 'A3'), (8, 'C4'), (4, 'F#4')]

melody2 = [(4, 'rest'), (2, 'rest'), (4, 'rest'), (2, 'rest'), (4, 'rest'), (2, 'rest'), (4, 'rest'), (2, 'rest'), 
           (2, 'B4'), (4, 'D5'), (2, 'A4'), (8, 'G4'), (8, 'A4'), (2, 'B4'), (4, 'D5'), (2, 'A4'), (4, 'rest'), (2, 'B4'),
           (4, 'D5'), (2, 'A5'), (4, 'G5'), (2, 'D5'), (8, 'C5'), (8, 'B4'), (2, 'A4')]

chord = []

player(melody, melody2, chord, 100, 'sinusoid', 'sinusoid')

In [186]:
#Demo 2

melody = [(8, 'A5'), (8, 'B5'), (2, 'A6'), (8, 'G#6'), (8, 'G6'), (4, 'F#6'), (4, 'D6'), (4, 'E6'), (4, 'B5'), (4, 'D6'), (2, 'A5'), (4, 'F#5'), (2, 'F#5'), (4, 'rest'),
         (8, 'A5'), (8, 'B5'), (2, 'A6'), (8, 'G#6'), (8, 'G6'), (4, 'F#6'), (4, 'A6'), (4, 'B6'), (4, 'D7'), (4, 'G7'), (1, 'F#7')]

melody2 = []

chords = [(4, ['rest']), (1, ['C4', 'E4', 'G4', 'B4']), (2, ['rest']), (4, ['rest']), (1, ['B3', 'D4', 'F#4', 'A4']), (2, ['rest']), (4, ['rest']),
         (4, ['rest']), (1, ['C4', 'E4', 'G4', 'B4']), (2, ['rest']), (4, ['rest']), (1, ['B3', 'D4', 'F#4', 'A4'])]

player(melody, melody2, chords, 120, 'sinusoid', 'sinusoid', 'square', fm = True, Mod = 15, delay = True)

In [189]:
#Demo 3

melody = [(8, 'G4'), (8, 'G4'), (4, 'A4'), (4, 'G4'), (4, 'C5'), (2, 'B4'), 
         (8, 'G4'), (8, 'G4'), (4, 'A4'), (4, 'G4'), (4, 'D5'), (2, 'C5'), 
         (8, 'G4'), (8, 'G4'), (4, 'G5'), (4, 'E5'), (4, 'C5'), (4, 'B4'), (4, 'A4'),
         (8, 'F5'), (8, 'F5'), (4, 'E5'), (4, 'C5'), (4, 'D5'), (2, 'C5')]

melody2 =[]

chords = [(4, ['rest']), (2, ['C3', 'E3', 'G3']), (4, ['rest']), (2, ['B2', 'D3', 'G3']), (4, ['rest']), 
          (2, ['B2', 'D3', 'G3']), (4, ['rest']), (2, ['C3', 'E3', 'G3']), (4, ['rest']), (2, ['C3', 'E3', 'G3']), (4, ['rest']),
          (2, ['C3', 'F3', 'A3']), (4, ['rest']), (2, ['C3', 'E3', 'G3']), (4, ['B2', 'D3', 'G3']), (2, ['C3', 'E3', 'G3'])]

player(melody, melody2, chords, 120, fm = True, Mod = 40, delay = True, offset_ms = 100, repeat = 3, a = 1, d = 30, s = 0.5, r = 1)